# Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/lifestyle_and_environment/lifestyle_and_environment.parquet'
df_life = read_parquet(pfile)
print(df_life.shape)
df_life.head()

In [ ]:
df_life_metadata = pd.read_csv('/home/ec2-user/studies/hpp_datasets/lifestyle_and_environment/metadata/lifestyle_and_environment_data_dictionary.csv')
need_feature = df_life_metadata[df_life_metadata['field_type'].isin(['Categorical (single)', 'Integer', 'Datetime', 'Date', 'Text'])]['tabular_field_name']
df_life_metadata.head()

In [ ]:
df_life = df_life[need_feature.tolist() + ['participant_id', 'cohort', 'research_stage', 'array_index']]

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_life)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Data Filtering

In [ ]:
life_metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_date', 'collection_timestamp', 'timezone', 'hmo', 'data_source']
life_metadata_col = [col for col in life_metadata_col if col in df_life.columns]
df_life_metadata = df_life[life_metadata_col]
df_life_features = df_life.drop(columns=life_metadata_col)

In [ ]:
missing_ratio_per_feature = df_life_features.isnull().sum() / len(df_life_features)  # 计算每列的缺失值比例
feature_threshold = 0.3  # 设定特征缺失率阈值

# 剔除缺失率超过阈值的特征
df_life_filtered = df_life_features.loc[:, missing_ratio_per_feature <= feature_threshold]
print(df_life_filtered.shape)

# Imputation

In [ ]:
import numpy as np
df_life_filtered = df_life_filtered.astype(np.float32)
means = df_life_filtered.mean()
df_life_imputed = df_life_filtered.fillna(means)

# Save

In [ ]:
df_life_metadata.reset_index(drop=True, inplace=True)
df_life_imputed.reset_index(drop=True, inplace=True)
df_life = pd.concat([df_life_metadata, df_life_imputed], axis=1).dropna(axis=1)
pfile = 'pheno_data/202512/055-lifestyle_and_environment_imputed.parquet'
df_life.to_parquet(pfile, index=False)

In [ ]:
pfile = 'pheno_data/202512/055-lifestyle_and_environment_imputed.parquet'
df_life = pd.read_parquet(pfile)
print(df_life.shape)
df_life.head()